# Denoising Autoencoder (DAE) — IEEE-CIS Fraud Detection

**Architecture basis:** `architecture_justification_v2.md` (Denoising Autoencoder branch)  
**Position in pipeline:** Parallel encoder alongside TSGC (TCN+GAT), outputs 64-d embedding to Feature Fusion  
**Preprocessing:** Identical to `experiment_gat.ipynb` (same `cols`, same split)

## Design Rationale
- **Why DAE over standard AE?** Gaussian noise injection forces the model to learn the statistical manifold of normal transactions, making it robust to noisy inputs and sensitive to out-of-distribution fraud patterns (WDAE-GAN 2025, DDAE 2025).  
- **Why 64-d bottleneck?** Matches all other encoders (TCN, GAT, CaT-GNN) for symmetric fusion weights in the Gated Attention Fusion.  
- **Dual-use output:** (1) 64-d encoder embedding -> Gated Attention Fusion; (2) Reconstruction error -> anomaly signal.  
- **Training split:** Normal transactions only (isFraud==0) for reconstruction; embeddings extracted for ALL rows.

## Preprocessing (Identical to experiment_gat.ipynb)
Column definitions, feature engineering, and train/val split are shared across all encoders.

In [ ]:
# Install DGL — uncomment the correct line for your environment
# Kaggle (PyTorch 2.1, CUDA 12.1)
# !pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html -q
# CPU-only / local
# !pip install dgl -q
!pip install dgl -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import GATConv
import dgl.dataloading

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')
import os, json, gc, time
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'DGL version: {dgl.__version__}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Graph construction (GTAN methodology — arch_justification_v2 §4.2)
GRAPH_K_NEIGHBORS = 3        # K=3 past temporal neighbors per relation (matches GTAN edge_per_trans=3)
GRAPH_RELATION_COLS = ['uid', 'card1', 'P_emaildomain', 'DeviceInfo']
MAX_GROUP_SIZE = 5000        # Skip hub groups (e.g. gmail.com) to avoid degree imbalance

# GAT model (arch_justification_v2 §4.4)
HIDDEN_DIM = 64              # Per-head output dim → L1=64×4=256, L2=64×1=64
NUM_HEADS_L1 = 4             # 4 heads: matches CaT-GNN paper; required for downstream attention scoring
FEAT_DROP = 0.3
ATTN_DROP = 0.3

# Training hyperparameters — ALL from experiment_tcn.ipynb
BATCH_SIZE   = 512           # experiment_tcn: BATCH_SIZE = 512
LR           = 0.0005        # experiment_tcn: LEARNING_RATE = 0.0005
EPOCHS       = 200           # experiment_tcn: EPOCHS = 200
PATIENCE_ES  = 50            # experiment_tcn: EarlyStopping patience = 50 (monitor val_auc)
PATIENCE_LR  = 10            # experiment_tcn: ReduceLROnPlateau patience = 10
LR_FACTOR    = 0.5           # experiment_tcn: factor = 0.5
MIN_LR       = 1e-7          # experiment_tcn: min_lr = 1e-7
FAN_OUT      = [10, 5]       # Neighbor sampling fan-out per layer

# Paths
SAVE_DIR = 'gat_results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'models'), exist_ok=True)

print('Configuration:')
print(f'  [ARCH v2]  Graph K={GRAPH_K_NEIGHBORS}, Relations: {GRAPH_RELATION_COLS}')
print(f'  [ARCH v2]  GAT: L1={HIDDEN_DIM}×{NUM_HEADS_L1}h={HIDDEN_DIM*NUM_HEADS_L1}-d, L2={HIDDEN_DIM}×1h={HIDDEN_DIM}-d')
print(f'  [TCN REF]  lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}')
print(f'  [TCN REF]  EarlyStopping patience={PATIENCE_ES}, ReduceLROnPlateau patience={PATIENCE_LR}')

---
## 1. Data Loading & Preprocessing

Identical to `experiment_tcn.ipynb` — same column selection, encoding functions, UID construction, and aggregation features. The only difference: we **do not reshape to sequences**. GAT uses the flat 263-feature vector as node features.

We also retain the `uid` string column temporarily for graph construction before dropping it from node features.

In [ ]:
# Load raw data
X_train  = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
X_test   = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_transaction.csv')
train_id = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')
test_id  = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_identity.csv')

# Normalize identity column names: Kaggle ships id-01 to id-38 (dash) in some
# splits and id_01 to id_38 (underscore) in others. Standardize before merge.
for df in [train_id, test_id, X_train, X_test]:
    df.columns = [c.replace("-", "_") if c.startswith("id") else c
                  for c in df.columns]

# Merge identity on TransactionID column (correct approach for IEEE-CIS)
X_train = X_train.merge(train_id, on="TransactionID", how="left")
X_test  = X_test.merge(test_id,  on="TransactionID", how="left")

# Extract labels then reset both to clean 0-based index
y_train = X_train["isFraud"].copy()
del X_train["isFraud"]

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f"Train: {X_train.shape},  Test: {X_test.shape}")
print(f"Fraud rate: {y_train.mean()*100:.2f}%")
print(f"id_ cols in train: {sum(1 for c in X_train.columns if c.startswith('id_'))}  "
      f"id_ cols in test: {sum(1 for c in X_test.columns if c.startswith('id_'))}")

In [ ]:
# Column definitions (same as experiment_tcn) ─────────────────────────────────
# V-columns selected by NaN-correlation clustering (experiment_tcn exact list)
V_COLS  = [1,3,4,6,8,11]
V_COLS += [13,14,17,20,23,26,27,30]
V_COLS += [36,37,40,41,44,47,48]
V_COLS += [54,56,59,62,65,67,68,70]
V_COLS += [76,78,80,82,86,88,89,91]
V_COLS += [107,108,111,115,117,120,121,123]
V_COLS += [124,127,129,130,136]
V_COLS += [138,139,142,147,156,162]
V_COLS += [165,160,166]
V_COLS += [178,176,173,182]
V_COLS += [187,203,205,207,215]
V_COLS += [169,171,175,180,185,188,198,210,209]
V_COLS += [218,223,224,226,228,229,235]
V_COLS += [220,221,234,238,250,271]          # present in experiment_tcn, was missing here
V_COLS += [240,258,257,253,252,260,261]
V_COLS += [264,266,267,274,277]
V_COLS += [294,284,285,286,291,297]
V_COLS += [303,305,307,309,310,320]
V_COLS += [281,283,289,296,301,314]

# Keep only selected V columns — drop the rest
all_cols = list(X_train.columns)
v_to_drop = [c for c in all_cols if c.startswith('V') and c not in [f'V{i}' for i in V_COLS]]
X_train.drop(columns=v_to_drop, inplace=True)
X_test.drop(columns=v_to_drop, inplace=True)

# Categorical columns (experiment_tcn str_type list — underscored after normalization)
CAT_COLS = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'id_12','id_15','id_16','id_23','id_27','id_28','id_29',
    'id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38',
    'DeviceType','DeviceInfo'
]

print(f'V_COLS count: {len(set(V_COLS))}')
print(f'Shape after V-col drop: {X_train.shape}')

In [ ]:
# Sort by TransactionDT — critical for GTAN temporal graph construction.
# Attach y_train as column so both sort together; avoids index-alignment issues.
X_train['__label__'] = y_train.values
X_train = X_train.sort_values('TransactionDT').reset_index(drop=True)
y_train = X_train.pop('__label__').astype('int8')
X_test  = X_test.sort_values('TransactionDT').reset_index(drop=True)

# day column (used for UID and D-transform)
X_train['day'] = X_train['TransactionDT'] / np.float32(24 * 60 * 60)
X_test['day']  = X_test['TransactionDT']  / np.float32(24 * 60 * 60)

# D-column transform — experiment_tcn exact approach:
#   formula : D_new = D_old - day   (same sign as experiment_tcn)
#   SKIP D1, D2, D3, D5, D9 — these keep their original values.
#   D1 in particular must NOT be transformed: UID uses floor(day - D1_orig).
#   If D1 were transformed to (day - D1_orig) first, UID would become floor(D1_orig) — wrong grouping.
D_SKIP = {1, 2, 3, 5, 9}
for i in range(1, 16):
    col = f'D{i}'
    if i in D_SKIP:
        continue          # keep original value
    if col in X_train.columns:
        X_train[col] = X_train[col] - X_train['day']
    if col in X_test.columns:
        X_test[col]  = X_test[col]  - X_test['day']

# DT_M — experiment_tcn exact formula (calendar month, not 30-day approximation)
import datetime
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
for df in [X_train, X_test]:
    dt_series = df['TransactionDT'].apply(
        lambda x: START_DATE + datetime.timedelta(seconds=x))
    df['DT_M'] = (dt_series.dt.year - 2017) * 12 + dt_series.dt.month

# Factorize categoricals + shift numerics positive, NaN -> -1
SKIP = {'TransactionAmt', 'TransactionDT', 'day', 'DT_M'}
shared_cols = set(X_train.columns) & set(X_test.columns)

for f in list(X_train.columns):
    if (str(X_train[f].dtype) == 'category') or (X_train[f].dtype == 'object'):
        if f in shared_cols:
            df_comb = pd.concat([X_train[f], X_test[f]], axis=0)
            df_comb, _ = df_comb.factorize(sort=True)
            if df_comb.max() > 32000:
                X_train[f] = df_comb[:len(X_train)].astype('int32')
                X_test[f]  = df_comb[len(X_train):].astype('int32')
            else:
                X_train[f] = df_comb[:len(X_train)].astype('int16')
                X_test[f]  = df_comb[len(X_train):].astype('int16')
        else:
            X_train[f], _ = X_train[f].factorize(sort=True)
            X_train[f] = X_train[f].astype('int16')
    elif f not in SKIP:
        if f in shared_cols:
            mn = np.min((X_train[f].min(), X_test[f].min()))
            X_train[f] -= np.float32(mn)
            X_test[f]  -= np.float32(mn)
            X_train[f].fillna(-1, inplace=True)
            X_test[f].fillna(-1, inplace=True)
        else:
            mn = X_train[f].min()
            X_train[f] -= np.float32(mn)
            X_train[f].fillna(-1, inplace=True)

print(f'Sorted + preprocessed. X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Fraud rate after sort: {y_train.mean()*100:.2f}%  (should be unchanged)')
print(f'DT_M range: {X_train["DT_M"].min()} to {X_train["DT_M"].max()}  (should be ~12 to 24)')

In [ ]:
# ── Encoding helper functions (same as experiment_tcn) ─────────────────────────
def encode_LE(col, verbose=True, df1=None, df2=None):
    if df1 is None: df1 = X_train
    if df2 is None: df2 = X_test
    df_comb = pd.concat([df1[col], df2[col]], axis=0)
    df_comb, _ = df_comb.factorize(sort=True)
    if df_comb.max() > 32000:
        df1[col] = df_comb[:len(df1)].astype('int32')
        df2[col] = df_comb[len(df1):].astype('int32')
    else:
        df1[col] = df_comb[:len(df1)].astype('int16')
        df2[col] = df_comb[len(df1):].astype('int16')
    if verbose: print(col, ', ', end='')

def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col], df2[col]])
        vc = df.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        nm = col + '_FE'
        df1[nm] = df1[col].map(vc).astype('float32')
        df2[nm] = df2[col].map(vc).astype('float32')
        print(nm, ', ', end='')

def encode_AG(main_columns, uids, aggregations=['mean'], train_df=None, test_df=None,
              fillna=True, usena=False):
    if train_df is None: train_df = X_train
    if test_df  is None: test_df  = X_test
    for main_column in main_columns:
        for col in uids:
            for agg_type in aggregations:
                new_col_name = main_column + '_' + col + '_' + agg_type
                temp_df = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]])
                if usena:
                    temp_df.loc[temp_df[main_column] == -1, main_column] = np.nan
                temp_df = temp_df.groupby([col])[main_column].agg([agg_type]).reset_index()
                temp_df = temp_df.rename(columns={agg_type: new_col_name})
                temp_df.index = list(temp_df[col])
                temp_df = temp_df[new_col_name].to_dict()
                train_df[new_col_name] = train_df[col].map(temp_df).astype('float32')
                test_df[new_col_name]  = test_df[col].map(temp_df).astype('float32')
                if fillna:
                    train_df[new_col_name].fillna(-1, inplace=True)
                    test_df[new_col_name].fillna(-1, inplace=True)
                print("'" + new_col_name + "'", ', ', end='')

def encode_AG2(main_columns, uids, train_df=None, test_df=None):
    if train_df is None: train_df = X_train
    if test_df  is None: test_df  = X_test
    for main_column in main_columns:
        for col in uids:
            comb = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]], axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            train_df[col + '_' + main_column + '_ct'] = train_df[col].map(mp).astype('float32')
            test_df[col + '_' + main_column + '_ct']  = test_df[col].map(mp).astype('float32')
            print(col + '_' + main_column + '_ct, ', end='')

def encode_CB(col1, col2, df1=None, df2=None):
    if df1 is None: df1 = X_train
    if df2 is None: df2 = X_test
    nm = col1 + '_' + col2
    df1[nm] = df1[col1].astype(str) + '_' + df1[col2].astype(str)
    df2[nm] = df2[col1].astype(str) + '_' + df2[col2].astype(str)
    encode_LE(nm, verbose=False)

print('Encoding functions defined.')

In [ ]:
# ── Feature engineering (same as experiment_tcn) ──────────────────────────────
# Cents feature
X_train['cents'] = (X_train['TransactionAmt'] - np.floor(X_train['TransactionAmt'])).astype('float32')
X_test['cents']  = (X_test['TransactionAmt']  - np.floor(X_test['TransactionAmt'])).astype('float32')

# Combined columns
encode_CB('card1', 'addr1')
encode_CB('card1_addr1', 'P_emaildomain')

# UID construction: card1_addr1 + floor(day - D1)
X_train['uid'] = (X_train['card1_addr1'].astype(str) + '_' +
                  np.floor(X_train['day'] - X_train['D1']).astype(str))
X_test['uid']  = (X_test['card1_addr1'].astype(str) + '_' +
                  np.floor(X_test['day']  - X_test['D1']).astype(str))

# Save raw uid & relation cols for graph construction BEFORE they get modified
uid_train    = X_train['uid'].copy()
card1_train  = X_train['card1'].copy()
# P_emaildomain and DeviceInfo are already label-encoded integers — fine for groupby
pemail_train = X_train['P_emaildomain'].copy()
device_train = X_train['DeviceInfo'].copy()

print('UID and relation columns saved for graph construction.')
print(f'Unique UIDs: {uid_train.nunique():,}')

In [ ]:
# ── Frequency, aggregation, count-unique encodings (same as experiment_tcn) ───
print('Frequency encoding:')
encode_FE(X_train, X_test, ['addr1', 'card1', 'card2', 'card3', 'P_emaildomain'])
encode_FE(X_train, X_test, ['card1_addr1', 'card1_addr1_P_emaildomain'])
encode_FE(X_train, X_test, ['uid'])

print('\nAggregation features:')
encode_AG(
    ['TransactionAmt', 'D9', 'D11'],
    ['card1', 'card1_addr1', 'card1_addr1_P_emaildomain'],
    ['mean', 'std'], usena=True
)
encode_AG(
    ['TransactionAmt', 'D4', 'D9', 'D10', 'D15'],
    ['uid'], ['mean', 'std'], fillna=True, usena=True
)
encode_AG(
    ['C' + str(x) for x in range(1, 15) if x != 3],
    ['uid'], ['mean'], fillna=True, usena=True
)
encode_AG(
    ['M' + str(x) for x in range(1, 10)],
    ['uid'], ['mean'], fillna=True, usena=True
)
encode_AG(['C14'], ['uid'], ['std'], fillna=True, usena=True)

print('\nCount-unique features:')
encode_AG2(['P_emaildomain', 'dist1', 'DT_M', 'id_02', 'cents'], ['uid'])
encode_AG2(['C13', 'V314'], ['uid'])
encode_AG2(['V127', 'V136', 'V309', 'V307', 'V320'], ['uid'])

# Outsider feature
X_train['outsider15'] = (np.abs(X_train['D1'] - X_train['D15']) > 3).astype('int8')
X_test['outsider15']  = (np.abs(X_test['D1']  - X_test['D15']) > 3).astype('int8')

print('\nDone.')

In [ ]:
# ── Final column selection (same drop list as experiment_tcn) ─────────────────
cols = list(X_train.columns)
DROP_COLS = (
    ['TransactionDT', 'TransactionID'] +
    ['D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14'] +
    ['DT_M', 'day', 'uid'] +
    ['C3', 'M5', 'id_08', 'id_33'] +
    ['card4', 'id_07', 'id_14', 'id_21', 'id_30', 'id_32', 'id_34'] +
    ['id_' + str(x) for x in range(22, 28)]
)
cols = [c for c in cols if c in X_train.columns and c not in DROP_COLS]

X_gat = X_train[cols].copy().fillna(-1).astype('float32').values  # (590540, n_features)
y_gat = y_train.values.astype('int64')

IN_DIM = X_gat.shape[1]
print(f'Node feature matrix: {X_gat.shape}  →  IN_DIM = {IN_DIM}')
print(f'Labels: {y_gat.shape}, fraud rate = {y_gat.mean()*100:.2f}%')

In [ ]:
# ── Stratified split on node IDs (transductive: all nodes in one graph) ────────
all_nids = np.arange(len(y_gat))
train_nids, val_nids = train_test_split(
    all_nids, test_size=0.2, stratify=y_gat, random_state=42
)
train_nids = torch.LongTensor(train_nids)
val_nids   = torch.LongTensor(val_nids)

print(f'Train nodes: {len(train_nids):,} | Val nodes: {len(val_nids):,}')
print(f'Train fraud: {y_gat[train_nids].mean()*100:.2f}% | '
      f'Val fraud: {y_gat[val_nids].mean()*100:.2f}%')

# StandardScaler fitted on train nodes only
scaler = StandardScaler()
X_gat[train_nids.numpy()] = scaler.fit_transform(X_gat[train_nids.numpy()])
X_gat[val_nids.numpy()]   = scaler.transform(X_gat[val_nids.numpy()])

node_features = torch.FloatTensor(X_gat)
node_labels   = torch.LongTensor(y_gat)

print(f'Feature tensor: {node_features.shape}, Labels tensor: {node_labels.shape}')

In [ ]:
# -- DAE Imports & Configuration ------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, precision_recall_curve,
    f1_score, confusion_matrix, accuracy_score,
    precision_score, recall_score, roc_curve)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, time, pickle

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = 'dae_results'
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)

# Architecture
INPUT_DIM    = len(cols)        # 263
HIDDEN_DIMS  = [200, 128]       # 263->200->128->64
LATENT_DIM   = 64               # matches TCN/GAT/CaT-GNN

# Training
NOISE_FACTOR = 0.15             # Gaussian noise std
BATCH_SIZE   = 1024
EPOCHS       = 100
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 15

print(f'Device:      {DEVICE}')
print(f'Input dim:   {INPUT_DIM}')
print(f'Architecture: {INPUT_DIM} -> {HIDDEN_DIMS[0]} -> {HIDDEN_DIMS[1]} -> {LATENT_DIM} -> {HIDDEN_DIMS[1]} -> {HIDDEN_DIMS[0]} -> {INPUT_DIM}')
print(f'Noise factor: {NOISE_FACTOR}')

In [ ]:
# -- Prepare tabular feature matrices (correct variable names from GAT preprocessing) --
# train_nids / val_nids are torch.LongTensor from the split cell
# X_gat is already scaled by GAT scaler — we re-extract raw from X_train DataFrame
train_idx_np = train_nids.numpy()
val_idx_np   = val_nids.numpy()

# Raw unscaled features — X_train DataFrame is still available
X_raw      = X_train[cols].fillna(-1).astype('float32').values   # (590540, 263)
X_test_raw = X_test[cols].fillna(-1).astype('float32').values    # (N_test,  263)

# Labels
y_train_arr = y_gat[train_idx_np]
y_val_arr   = y_gat[val_idx_np]

# Apply DAE's own StandardScaler (separate from GAT scaler)
dae_scaler  = StandardScaler()
X_train_sc  = dae_scaler.fit_transform(X_raw[train_idx_np]).astype('float32')
X_val_sc    = dae_scaler.transform(X_raw[val_idx_np]).astype('float32')
X_test_sc   = dae_scaler.transform(X_test_raw).astype('float32')

# DAE trains ONLY on normal (benign) transactions
normal_mask = (y_train_arr == 0)
X_normal    = X_train_sc[normal_mask]

print(f'Train: {X_train_sc.shape} | fraud: {y_train_arr.sum():,} ({y_train_arr.mean()*100:.1f}%)')
print(f'Val:   {X_val_sc.shape}   | fraud: {y_val_arr.sum():,} ({y_val_arr.mean()*100:.1f}%)')
print(f'Test:  {X_test_sc.shape}')
print(f'Normal (DAE training set): {X_normal.shape}')

In [ ]:
# -- Improved DAE: Feature Masking + Larger Encoder + Classifier Head -----
# Upgrade 1: Feature masking (30% zeroed) >> Gaussian noise for tabular data
# Upgrade 2: Larger encoder 263->512->256->128->64 (more capacity)
# Upgrade 3: Classifier head for Phase 2 semi-supervised fine-tuning
# Upgrade 4: Anomaly score = recon_error + 0.1 * ||z||^2 (combined signal)

class ImprovedDAE(nn.Module):
    def __init__(self, input_dim=263, latent_dim=64, mask_ratio=0.30, dropout=0.3):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, latent_dim),
            nn.LayerNorm(latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, input_dim),
        )
        # Classifier head for Phase 2 semi-supervised training
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def mask_features(self, x):
        mask = torch.bernoulli(torch.full(x.shape, 1 - self.mask_ratio, device=x.device))
        return x * mask

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        x_in    = self.mask_features(x) if self.training else x
        z       = self.encoder(x_in)
        x_recon = self.decoder(z)
        return x_recon, z

    def anomaly_score(self, x, lambda_z=0.1):
        x_recon, z = self.forward(x)
        recon_err   = ((x_recon - x) ** 2).mean(dim=1)
        latent_norm = (z ** 2).mean(dim=1)
        return recon_err + lambda_z * latent_norm


dae = ImprovedDAE(input_dim=INPUT_DIM, latent_dim=LATENT_DIM,
                  mask_ratio=0.30, dropout=0.3).to(DEVICE)
total_params = sum(p.numel() for p in dae.parameters() if p.requires_grad)
print(f'Improved DAE parameters: {total_params:,}')
print(f'Encoder: {INPUT_DIM}->512->256->128->{LATENT_DIM}')
print(f'Decoder: {LATENT_DIM}->128->256->512->{INPUT_DIM}')
print(f'Mask ratio: 30% features zeroed per sample')

In [ ]:
# -- Phase 1: Unsupervised DAE (normal transactions, feature masking) ------
X_normal_t  = torch.FloatTensor(X_normal)
X_train_all = torch.FloatTensor(X_train_sc)
X_val_t     = torch.FloatTensor(X_val_sc)
y_train_t   = torch.FloatTensor(y_train_arr)

train_ds_p1 = torch.utils.data.TensorDataset(X_normal_t)
loader_p1   = torch.utils.data.DataLoader(train_ds_p1, batch_size=BATCH_SIZE, shuffle=True)

criterion_mse = nn.MSELoss()
optimizer_p1  = torch.optim.AdamW(dae.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_p1  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=EPOCHS)

history_p1   = {'loss':[], 'auc':[], 'score_normal':[], 'score_fraud':[]}
best_auc_p1, best_state, patience_ctr = 0.0, None, 0
start = time.time()

print('='*60)
print('PHASE 1: UNSUPERVISED DAE (Feature Masking)')
print('='*60)
print(f'Training on {len(X_normal):,} normal transactions')

for epoch in range(1, EPOCHS + 1):
    dae.train()
    epoch_loss = 0.0
    for (xb,) in loader_p1:
        xb = xb.to(DEVICE)
        optimizer_p1.zero_grad()
        recon, _ = dae(xb)
        loss = criterion_mse(recon, xb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dae.parameters(), 1.0)
        optimizer_p1.step()
        epoch_loss += loss.item()
    epoch_loss /= len(loader_p1)
    scheduler_p1.step()

    dae.eval()
    with torch.no_grad():
        scores = dae.anomaly_score(X_val_t.to(DEVICE)).cpu().numpy()
    val_auc      = roc_auc_score(y_val_arr, scores)
    norm_score   = scores[y_val_arr == 0].mean()
    fraud_score  = scores[y_val_arr == 1].mean()

    history_p1['loss'].append(epoch_loss)
    history_p1['auc'].append(val_auc)
    history_p1['score_normal'].append(float(norm_score))
    history_p1['score_fraud'].append(float(fraud_score))

    if val_auc > best_auc_p1:
        best_auc_p1  = val_auc
        best_state   = {k: v.clone() for k, v in dae.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Loss: {epoch_loss:.5f} | AUC: {val_auc:.4f} | Score[N]: {norm_score:.4f} | Score[F]: {fraud_score:.4f}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'Phase 1 done in {(time.time()-start)/60:.1f} min | Best AUC: {best_auc_p1:.4f}')
dae.load_state_dict(best_state)

In [ ]:
# -- Training Curves -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], 'b-', linewidth=2)
axes[0].set_title('Training Loss (MSE)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_auc'], 'g-', linewidth=2)
axes[1].axhline(best_auc, color='r', linestyle='--', label=f'Best: {best_auc:.4f}')
axes[1].set_title('Validation AUC (Reconstruction Error)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['val_recon_normal'], 'b-', linewidth=2, label='Normal')
axes[2].plot(history['val_recon_fraud'],  'r-', linewidth=2, label='Fraud')
axes[2].set_title('Mean Reconstruction Error per Class', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MSE'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/dae_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# -- Phase 2: Semi-Supervised Fine-Tuning with Fraud Labels ----------------
EPOCHS_P2 = 40
LR_P2     = 5e-4
pos_w     = torch.tensor([(y_train_arr==0).sum() / max((y_train_arr==1).sum(), 1)]).to(DEVICE)

# Freeze decoder
for p in dae.decoder.parameters(): p.requires_grad = False

optimizer_p2  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, dae.parameters()), lr=LR_P2, weight_decay=1e-4)
criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)

train_ds_p2 = torch.utils.data.TensorDataset(X_train_all, y_train_t)
loader_p2   = torch.utils.data.DataLoader(train_ds_p2, batch_size=BATCH_SIZE, shuffle=True)

best_auc_p2, best_state_p2 = 0.0, None
history_p2  = {'loss':[], 'auc':[]}
start2 = time.time()

print('='*60)
print('PHASE 2: SEMI-SUPERVISED FINE-TUNING')
print('='*60)
print(f'Decoder frozen | Encoder + classifier head training')
print(f'pos_weight: {pos_w.item():.1f}x')

for epoch in range(1, EPOCHS_P2 + 1):
    dae.train()
    epoch_loss = 0.0
    for xb, yb in loader_p2:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_p2.zero_grad()
        z     = dae.encode(xb)
        logit = dae.classifier(z).squeeze()
        loss  = criterion_bce(logit, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dae.parameters(), 1.0)
        optimizer_p2.step()
        epoch_loss += loss.item()
    epoch_loss /= len(loader_p2)

    dae.eval()
    with torch.no_grad():
        z_val  = dae.encode(X_val_t.to(DEVICE))
        probs  = torch.sigmoid(dae.classifier(z_val).squeeze()).cpu().numpy()
    val_auc = roc_auc_score(y_val_arr, probs)
    history_p2['loss'].append(epoch_loss)
    history_p2['auc'].append(val_auc)

    if val_auc > best_auc_p2:
        best_auc_p2  = val_auc
        best_state_p2 = {k: v.clone() for k, v in dae.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS_P2} | Loss: {epoch_loss:.5f} | Val AUC: {val_auc:.4f}')

dae.load_state_dict(best_state_p2)
torch.save(dae.state_dict(), f'{SAVE_DIR}/models/dae_best.pt')
for p in dae.decoder.parameters(): p.requires_grad = True
print(f'Phase 2 done in {(time.time()-start2)/60:.1f} min | Best AUC: {best_auc_p2:.4f}')
print(f'Gain over Phase 1: {best_auc_p2 - best_auc_p1:+.4f}')

In [ ]:
# -- Final Evaluation: Phase 1 & Phase 2 ----------------------------------
print('='*60); print('FINAL EVALUATION'); print('='*60)
dae.eval()
with torch.no_grad():
    scores_p1 = dae.anomaly_score(X_val_t.to(DEVICE)).cpu().numpy()
    z_val     = dae.encode(X_val_t.to(DEVICE))
    probs_p2  = torch.sigmoid(dae.classifier(z_val).squeeze()).cpu().numpy()
    val_recon, _ = dae(X_val_t.to(DEVICE))
    recon_errors = ((val_recon.cpu() - X_val_t)**2).mean(dim=1).numpy()

auc_p1 = roc_auc_score(y_val_arr, scores_p1)
auc_p2 = roc_auc_score(y_val_arr, probs_p2)

precisions, recalls, thresholds = precision_recall_curve(y_val_arr, probs_p2)
f1s = 2*precisions*recalls/(precisions+recalls+1e-8)
best_thresh = thresholds[np.argmax(f1s)]
y_pred = (probs_p2 >= best_thresh).astype(int)

f1   = f1_score(y_val_arr, y_pred)
acc  = accuracy_score(y_val_arr, y_pred)
pre  = precision_score(y_val_arr, y_pred)
rec  = recall_score(y_val_arr, y_pred)
tn, fp, fn, tp = confusion_matrix(y_val_arr, y_pred).ravel()
spec = tn/(tn+fp)
separation = recon_errors[y_val_arr==1].mean()/recon_errors[y_val_arr==0].mean()

print(f'Phase 1 AUC (unsupervised): {auc_p1:.4f}')
print(f'Phase 2 AUC (semi-sup):     {auc_p2:.4f}')
print(f'F1-Score:    {f1:.4f}')
print(f'Accuracy:    {acc:.4f}')
print(f'Precision:   {pre:.4f}')
print(f'Recall:      {rec:.4f}')
print(f'Specificity: {spec:.4f}')
print(f'CM: TN={tn:,} FP={fp:,} FN={fn:,} TP={tp:,}')
print(f'Recon separation: {separation:.2f}x')

In [ ]:
# -- Evaluation Plots ------------------------------------------------------
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clip_pct = np.percentile(recon_errors, 99)
axes[0].hist(recon_errors[y_val_arr==0], bins=80, alpha=0.7, color='steelblue', label='Normal', density=True)
axes[0].hist(recon_errors[y_val_arr==1], bins=80, alpha=0.7, color='crimson', label='Fraud', density=True)
axes[0].set_xlabel('Reconstruction Error'); axes[0].set_title('Reconstruction Error Distribution', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_xlim([0, clip_pct])

fpr1, tpr1, _ = roc_curve(y_val_arr, scores_p1)
fpr2, tpr2, _ = roc_curve(y_val_arr, probs_p2)
axes[1].plot(fpr1, tpr1, 'b--', linewidth=2, label=f'Phase 1 Unsup. (AUC={auc_p1:.4f})')
axes[1].plot(fpr2, tpr2, 'g-',  linewidth=2.5, label=f'Phase 2 Semi-sup. (AUC={auc_p2:.4f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC: Phase 1 vs Phase 2', fontweight='bold'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].plot(history_p2['auc'], 'g-', linewidth=2)
axes[2].axhline(best_auc_p2, color='r', linestyle='--', label=f'Best: {best_auc_p2:.4f}')
axes[2].set_title('Phase 2 AUC Curve', fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/dae_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# -- Extract 64-d Encoder Embeddings for Gated Attention Fusion ------------
print('='*60)
print('EXTRACTING 64-d ENCODER EMBEDDINGS')
print('='*60)

dae.eval()

def extract_embeddings(X_np, batch_size=2048):
    dataset = TensorDataset(torch.FloatTensor(X_np))
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    embs = []
    with torch.no_grad():
        for (xb,) in loader:
            z = dae.encode(xb.to(DEVICE))
            embs.append(z.cpu().numpy())
    return np.vstack(embs)

dae_emb_train = extract_embeddings(X_train_sc)   # (N_train, 64)
dae_emb_val   = extract_embeddings(X_val_sc)     # (N_val,   64)
dae_emb_test  = extract_embeddings(X_test_sc)    # (N_test,  64)

print(f'Train embeddings: {dae_emb_train.shape}')
print(f'Val   embeddings: {dae_emb_val.shape}')
print(f'Test  embeddings: {dae_emb_test.shape}')

# Cosine separability check
from sklearn.preprocessing import normalize
emb_norm = normalize(dae_emb_val)
fraud_centroid  = emb_norm[y_val_arr == 1].mean(0)
normal_centroid = emb_norm[y_val_arr == 0].mean(0)
cos_sep = 1 - np.dot(fraud_centroid, normal_centroid)
print(f'Embedding cosine separation (fraud vs normal): {cos_sep:.4f}')
print('(Higher = embeddings are more separable in latent space)')

In [ ]:
# -- Save Model, Scaler, Embeddings, Results --------------------------------
with open(f'{SAVE_DIR}/models/dae_scaler.pkl', 'wb') as f:
    pickle.dump(dae_scaler, f)

np.save(f'{SAVE_DIR}/dae_emb_train.npy',    dae_emb_train)
np.save(f'{SAVE_DIR}/dae_emb_val.npy',      dae_emb_val)
np.save(f'{SAVE_DIR}/dae_emb_test.npy',     dae_emb_test)
np.save(f'{SAVE_DIR}/recon_errors_val.npy', recon_errors)

results = {
    'auc_recon':   float(auc_recon),
    'threshold':   float(best_thresh),
    'f1':          float(f1),
    'precision':   float(pre),
    'recall':      float(rec),
    'specificity': float(spec),
    'accuracy':    float(acc),
    'recon_error': {
        'normal_mean':      float(recon_errors[y_val_arr==0].mean()),
        'normal_std':       float(recon_errors[y_val_arr==0].std()),
        'fraud_mean':       float(recon_errors[y_val_arr==1].mean()),
        'fraud_std':        float(recon_errors[y_val_arr==1].std()),
        'separation_ratio': float(separation),
    },
    'model_config': {
        'input_dim':    INPUT_DIM,
        'hidden_dims':  HIDDEN_DIMS,
        'latent_dim':   LATENT_DIM,
        'noise_factor': NOISE_FACTOR,
        'total_params': total_params,
    }
}
with open(f'{SAVE_DIR}/dae_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('='*60)
print('SAVED FILES')
print('='*60)
print(f'  {SAVE_DIR}/models/dae_best.pt')
print(f'  {SAVE_DIR}/models/dae_scaler.pkl')
print(f'  {SAVE_DIR}/dae_emb_train.npy     {dae_emb_train.shape}')
print(f'  {SAVE_DIR}/dae_emb_val.npy       {dae_emb_val.shape}')
print(f'  {SAVE_DIR}/dae_emb_test.npy      {dae_emb_test.shape}')
print(f'  {SAVE_DIR}/dae_results.json')

In [ ]:
# -- Encoder Comparison ---------------------------------------------------
import pandas as pd
df = pd.DataFrame({
    'Model':     ['TCN','GAT','CaT-GNN','DAE P1','DAE P2'],
    'AUC':       [0.9688, 0.9374, 0.9503, round(auc_p1,4), round(auc_p2,4)],
    'F1':        [0.4806, 0.6288, 0.6783, '---', round(f1,4)],
    'Precision': [0.3289, 0.6877, 0.7142, '---', round(pre,4)],
    'Recall':    [0.8918, 0.5792, 0.6458, '---', round(rec,4)],
    'Mode':      ['Sup','Sup','Sup','Unsup','Semi-sup'],
})
print(df.to_string(index=False))
print()
print('NEXT: Update experiment_fusion.ipynb')
print('  Extend GatedFusion: 3 inputs (TCN,GAT,CaT-GNN) -> 4 inputs (+DAE)')
print('  Gate network: 384->128->3 becomes 256->128->4')